# 02 - Preprocessing (Stage 2)
**Owner:** Aiman | **Updated 7 Nov:** reads the complete data from Drive and saves in resume-safe shards

What this notebook does:
1. Reads the folder paths and expected counts from `raw_inventory.json` (written by notebook 01)
2. Crop the black border, pad to a square, resize to **512**, **CLAHE**, circular mask
3. **Patient-aware split 70/15/15** (frozen: one split file for the whole team)
4. Processes the images in **shards of 2,000**. Each finished shard (a zip and a small CSV) is saved to Drive at once, so a Colab disconnect loses at most one shard. Run the notebook again and finished shards are skipped.
5. Builds `manifest.csv` and checks everything

**How to run**
- First run with `MODE = "smoke"` (about 10 minutes, writes nothing to Drive).
- If the smoke run is clean, change to `MODE = "full"` and run all cells. Expect about **3.5 hours** in total (about 0.32 s per image, 38,788 images). You can stop at any time and run again later. Set `MAX_SHARDS_THIS_RUN = 5`, for example, to do only 5 shards per session.
- APTOS is processed first (about 20 minutes), then EyePACS.

**What we do NOT do here:** no augmentation, no SMOTE, no oversampling. The files keep the natural class distribution (75% No DR). Focal Loss and SMOTE are compared later on the same files.

In [ ]:
# ===== X-Retina bootstrap: paste as the FIRST cell of every notebook =====
# Colab Secrets needed: GITHUB_TOKEN (or GITHUB-TOKEN). Optional: GIT_NAME, GIT_EMAIL, KAGGLE_USERNAME, KAGGLE_KEY
GITHUB_USER = "aimenkhan1"
REPO_NAME   = "X-Retina---Final-Year-Project-"
BRANCH      = "main"

import os, sys, json, subprocess
from google.colab import drive, userdata

def _secret(*names, default=None):
    for n in names:
        try:
            v = userdata.get(n)
            if v:
                return v
        except Exception:
            pass
    return default

GIT_NAME  = _secret("GIT_NAME",  default="X-Retina Team")
GIT_EMAIL = _secret("GIT_EMAIL", default="noreply@users.noreply.github.com")

# 1) Drive + shared folder
drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/FYP(Gr2-Fall 2026)/XRetina_Shared"
assert os.path.isdir(DRIVE_ROOT), (
    f"Shared folder not found at {DRIVE_ROOT}. Add a shortcut named 'XRetina_Shared' to My Drive.")
for _sub in ("processed", "splits", "checkpoints", "outputs"):
    os.makedirs(f"{DRIVE_ROOT}/{_sub}", exist_ok=True)
os.environ["XRETINA_DRIVE_ROOT"] = DRIVE_ROOT
os.makedirs("/content/data/raw", exist_ok=True)
os.environ["XRETINA_RAW_DIR"] = "/content/data/raw"

# 2) Clone or update the repo (token is hidden from error messages)
_token = _secret("GITHUB_TOKEN", "GITHUB-TOKEN")
assert _token, "Add GITHUB_TOKEN (or GITHUB-TOKEN) in Colab Secrets and switch on Notebook access."
REPO_DIR = f"/content/{REPO_NAME}"
_url = f"https://{_token}@github.com/{GITHUB_USER}/{REPO_NAME}.git"

def _run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd[:4])} failed:\n{r.stderr.replace(_token, '***')}")
    return r.stdout.strip()

if not os.path.isdir(REPO_DIR):
    _run(["git", "clone", _url, REPO_DIR])
_run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", _url])
_run(["git", "-C", REPO_DIR, "fetch", "origin"])
try:
    _run(["git", "-C", REPO_DIR, "checkout", BRANCH])
except RuntimeError:
    _run(["git", "-C", REPO_DIR, "checkout", "-b", BRANCH])  # first time on a new branch
try:
    _run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH])
except RuntimeError as e:
    print("Pull skipped (new local branch or local changes):", str(e).splitlines()[-1])
_run(["git", "-C", REPO_DIR, "config", "user.name", GIT_NAME])
_run(["git", "-C", REPO_DIR, "config", "user.email", GIT_EMAIL])

# 3) Requirements
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO_DIR}/requirements.txt"], check=True)

# 4) Kaggle credentials (only needed to download APTOS)
try:
    os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
    _kp = os.path.expanduser("~/.kaggle/kaggle.json")
    with open(_kp, "w") as f:
        json.dump({"username": userdata.get("KAGGLE_USERNAME"), "key": userdata.get("KAGGLE_KEY")}, f)
    os.chmod(_kp, 0o600)
except Exception as e:
    print("Kaggle secrets not set (only needed to download APTOS):", e)

# 5) Make `import src...` work
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

def git_push(message):
    """Commit and push your work directly to the current branch."""
    _run(["git", "-C", REPO_DIR, "add", "-A"])
    print(_run(["git", "-C", REPO_DIR, "commit", "-m", message]))
    print(_run(["git", "-C", REPO_DIR, "push", "-u", "origin", BRANCH]))

from src.config import get_paths
from src.utils import seed_everything, get_device
PATHS = get_paths()
print("Repo:", REPO_DIR, "| Branch:", BRANCH, "| Device:", get_device())
print({k: str(v) for k, v in PATHS.items()})

## 1. Settings
Only this cell is edited by hand. All preprocessing numbers are written down here and saved to `preprocessing_params.json`, so the report can quote them.

**Where the settings come from (checked in the PDFs)**
- 512 cache, CLAHE clip 2.0 / tile 8, crop threshold: **our own choices**. None of [7], [8], [9], [11] gives CLAHE values. [11] only says "contrast enhancement".
- [9] Patni: center crop + contrast normalisation, resize 224. [7], [8]: resize 224 + normalisation only.
- Split: [7] 85/15, [8] 80/10/10, [9] 80/20, all random image-level. We split **by patient** because EyePACS has two eyes per person.

In [ ]:
# ===== SETTINGS (edit only this cell) =====
MODE = "smoke"           # "smoke": ~300 images per dataset, writes NOTHING to Drive
                         # "full" : all images, writes splits + shards + manifest to Drive
SEED          = 42
OUT_SIZE      = 512      # cache size (train later at 380, or 224 for a like-for-like run with [11])
CLAHE_CLIP    = 2.0      # our setting (not from [11])
CLAHE_TILE    = 8
DARK_THR      = 10       # gray level below this counts as background when cropping
JPEG_QUALITY  = 95
SPLIT         = {"train": 0.70, "val": 0.15, "test": 0.15}
N_WORKERS     = 8
SMOKE_N       = 300
SHARD_SIZE_FULL = 2000   # images per shard in full mode (smoke mode uses 100)
MAX_SHARDS_THIS_RUN = None   # None = no limit; 5 = stop after 5 new shards in this session
STRICT_COUNTS = True     # compare with the counts notebook 01 saved
FORCE_RESPLIT = False    # True only if the WHOLE team agrees to replace the frozen split
REPROCESS_FAILED_SHARDS = False   # True = redo shards that contain failed images

import time, json, os, zipfile, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
cv2.setNumThreads(1)

assert MODE in ("smoke", "full")
assert abs(sum(SPLIT.values()) - 1) < 1e-9
FULL = MODE == "full"
DRIVE      = Path(DRIVE_ROOT)
PROC_DIR   = DRIVE / "processed"
SPLIT_DIR  = DRIVE / "splits"
OUT_DIR    = DRIVE / "outputs"
SPLIT_FILE = SPLIT_DIR / "split_v1.csv"
LOCAL_OUT  = Path("/content/processed_full" if FULL else "/content/processed_smoke")
LOCAL_OUT.mkdir(parents=True, exist_ok=True)
SHARD_SIZE = SHARD_SIZE_FULL if FULL else 100
SHARD_DIR  = PROC_DIR if FULL else LOCAL_OUT / "shards"      # where finished shards are saved
META_DIR   = PROC_DIR if FULL else LOCAL_OUT                  # where manifest/params go

# Folder paths and expected counts come from notebook 01
INV_PATH = Path(PATHS["outputs"]) / "raw_inventory.json"
assert INV_PATH.exists(), "raw_inventory.json not found: run notebook 01 first."
INV = json.loads(INV_PATH.read_text())
assert "updated" in INV and "path" in INV.get("eyepacs", {}) and "path" in INV.get("aptos2019", {}), \
    "raw_inventory.json is in the OLD format: run the UPDATED notebook 01 first."
EYE_DIR   = Path(INV["eyepacs"]["path"])
APTOS_DIR = Path(INV["aptos2019"]["path"])
if FULL:
    assert INV["eyepacs"]["complete"] and INV["aptos2019"]["complete"], \
        "Notebook 01 says a dataset is incomplete. Fix that and run notebook 01 again."

PARAMS = dict(out_size=OUT_SIZE, clahe_clip=CLAHE_CLIP, clahe_tile=CLAHE_TILE,
              dark_threshold=DARK_THR, jpeg_quality=JPEG_QUALITY, split=SPLIT, seed=SEED,
              shard_size=SHARD_SIZE, inventory_updated=INV["updated"],
              steps=["crop black border", "pad to square (black)", "resize (INTER_AREA)",
                     "CLAHE on L channel of LAB", "circular mask"],
              opencv=cv2.__version__)
print(f"MODE = {MODE}  ->", "writes to Drive" if FULL else "smoke test, nothing is written to Drive")
print("EyePACS:", EYE_DIR)
print("APTOS  :", APTOS_DIR)
print("Inventory from:", INV["updated"])

## 2. Load labels
Both datasets are read straight from Drive (folders taken from `raw_inventory.json`). Nothing is downloaded. The numbers found here must match the numbers notebook 01 saved.

The first scan of about 39k files on Drive takes several minutes.

In [ ]:
from src.data.labels import load_labels, patient_id_from_name

def _check_counts(df, key):
    exp = INV[key]
    assert len(df) == exp["report"]["n_labeled"], \
        f"{key}: found {len(df)} labelled images, notebook 01 saw {exp['report']['n_labeled']}"
    assert df["label"].value_counts().sort_index().tolist() == list(exp["class_counts"].values()), \
        f"{key}: class counts differ from notebook 01"

# ---- EyePACS ----
eye_df, _ = load_labels(EYE_DIR)
eye_df = eye_df[["image_id", "label", "image_path"]].copy()
eye_df["dataset"] = "eyepacs"
_pid = eye_df["image_id"].map(patient_id_from_name)
assert _pid.isna().sum() == 0, "some EyePACS images have no patient id"
eye_df["patient_id"] = _pid.astype(str)
eye_df["eye"] = eye_df["image_id"].astype(str).str.rsplit("_", n=1).str[-1]
print("EyePACS:", len(eye_df), "images |", eye_df["patient_id"].nunique(), "patients")
if STRICT_COUNTS:
    _check_counts(eye_df, "eyepacs")
    assert eye_df["patient_id"].nunique() == INV["eyepacs"]["n_patients"]

# ---- APTOS ----
ap_df, _ = load_labels(APTOS_DIR)
ap_df = ap_df[["image_id", "label", "image_path"]].copy()
ap_df["dataset"] = "aptos"
ap_df["patient_id"] = ap_df["image_id"].astype(str)   # APTOS has no patient id: each image is its own group
ap_df["eye"] = "unknown"
print("APTOS:", len(ap_df), "images")
if STRICT_COUNTS:
    _check_counts(ap_df, "aptos2019")

all_df = pd.concat([eye_df, ap_df], ignore_index=True)
all_df["image_id"] = all_df["image_id"].astype(str)
if not FULL:   # smoke test: small random sample per dataset
    all_df = pd.concat([g.sample(min(SMOKE_N, len(g)), random_state=SEED)
                        for _, g in all_df.groupby("dataset")], ignore_index=True)
all_df = all_df.sort_values(["dataset", "image_id"]).reset_index(drop=True)   # fixed order = stable shards
print("\nImages used in this run:")
display(pd.crosstab(all_df["dataset"], all_df["label"], margins=True))

## 3. Preprocessing function
Steps, in this order:
1. **Crop** the black border around the retina
2. **Pad** to a square with black (so the retina is never stretched)
3. **Resize** to 512 x 512
4. **CLAHE** on the lightness channel only (colours stay natural)
5. **Circular mask** (black outside the circle)

The crop and padding numbers are saved in the manifest for every image. In FYP-II the IDRiD lesion masks must get exactly the same crop, so a lesion pixel at original position (x, y) lands at

`x_out = (x - crop_x0 + pad_left) * 512 / square_side` and `y_out = (y - crop_y0 + pad_top) * 512 / square_side`

In [ ]:
def preprocess_array(img):
    h0, w0 = img.shape[:2]
    # 1. crop the black border
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    mask = gray > DARK_THR
    rows = np.where(mask.mean(axis=1) > 0.01)[0]
    cols = np.where(mask.mean(axis=0) > 0.01)[0]
    if len(rows) == 0 or len(cols) == 0:          # blank image: keep everything
        y0, y1, x0, x1 = 0, h0, 0, w0
    else:
        y0, y1, x0, x1 = int(rows.min()), int(rows.max()) + 1, int(cols.min()), int(cols.max()) + 1
    img = img[y0:y1, x0:x1]
    # 2. pad to a square
    h, w = img.shape[:2]
    side = max(h, w)
    pt, pl = (side - h) // 2, (side - w) // 2
    img = cv2.copyMakeBorder(img, pt, side - h - pt, pl, side - w - pl,
                             cv2.BORDER_CONSTANT, value=(0, 0, 0))
    # 3. resize
    interp = cv2.INTER_AREA if side >= OUT_SIZE else cv2.INTER_CUBIC
    img = cv2.resize(img, (OUT_SIZE, OUT_SIZE), interpolation=interp)
    # 4. CLAHE on L channel
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    clahe = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=(CLAHE_TILE, CLAHE_TILE))
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    img = cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)
    # 5. circular mask
    circ = np.zeros((OUT_SIZE, OUT_SIZE), np.uint8)
    cv2.circle(circ, (OUT_SIZE // 2, OUT_SIZE // 2), OUT_SIZE // 2, 255, -1)
    img = cv2.bitwise_and(img, img, mask=circ)
    meta = dict(orig_w=int(w0), orig_h=int(h0), crop_x0=x0, crop_y0=y0, crop_x1=x1, crop_y1=y1,
                pad_top=int(pt), pad_left=int(pl), square_side=int(side))
    return img, meta

def preprocess(path):
    img = None
    for attempt in range(3):                       # Drive sometimes fails once: try again
        img = cv2.imread(str(path))
        if img is not None:
            break
        time.sleep(1.0 * (attempt + 1))
    if img is None:
        raise IOError(f"cannot read {path}")
    return preprocess_array(img)

### Look at 5 examples (one per grade)
Check in the picture: the retina fills the circle, it is not stretched, vessels and spots look clearer but not noisy. If grade 3 or 4 images look over-bright, lower `CLAHE_CLIP` in the settings cell.

In [ ]:
demo_src = all_df[all_df["dataset"] == "eyepacs"]
if len(demo_src) == 0 or not Path(demo_src["image_path"].iloc[0]).exists():
    print("EyePACS images not found: example figure skipped.")
else:
    demo = demo_src.groupby("label").sample(1, random_state=7).sort_values("label")
    fig, ax = plt.subplots(2, len(demo), figsize=(4 * len(demo), 8))
    t0 = time.time()
    for i, (_, r) in enumerate(demo.iterrows()):
        before = cv2.cvtColor(cv2.imread(str(r["image_path"])), cv2.COLOR_BGR2RGB)
        after, meta = preprocess(r["image_path"])
        ax[0, i].imshow(before); ax[0, i].set_title(f"Grade {r['label']} - before"); ax[0, i].axis("off")
        ax[1, i].imshow(cv2.cvtColor(after, cv2.COLOR_BGR2RGB)); ax[1, i].set_title("after"); ax[1, i].axis("off")
    print(f"{(time.time() - t0) / len(demo):.2f} s per image (one thread, first read from Drive).")
    print("The real speed (about 0.3 s per image) is printed after every shard below.")
    plt.tight_layout()
    if FULL:
        plt.savefig(OUT_DIR / "preprocessing_examples.png", dpi=100)
    plt.show()

## 4. Patient-aware split (70 / 15 / 15)
- EyePACS: a patient has a left and a right eye. Both eyes always go to the **same** split, otherwise the model sees the same person in train and test.
- Each patient is stratified by their **worst eye grade**, so every split keeps the same grade mix.
- APTOS has no patient id, so each image is its own group (same as [7], [8], [9]). We say this in the report.
- Each dataset is split **separately**, so results can be reported per dataset later.
- The split is **frozen**: the first full run saves `splits/split_v1.csv`. Every later run (and every teammate) loads that file. All experiments (baseline, Focal Loss, SMOTE, hybrid) use the same split.

In [ ]:
def patient_level_split(df, seed):
    pat = df.groupby("patient_id")["label"].max().rename("worst").reset_index()
    def _tts(frame, size):
        try:
            return train_test_split(frame, test_size=size, stratify=frame["worst"], random_state=seed)
        except ValueError:
            print("  WARNING: a grade has too few patients to stratify -> plain random split (fine in smoke test only)")
            return train_test_split(frame, test_size=size, random_state=seed)
    rest_size = SPLIT["val"] + SPLIT["test"]
    tr, rest = _tts(pat, rest_size)
    va, te = _tts(rest, SPLIT["test"] / rest_size)
    m = {p: "train" for p in tr["patient_id"]}
    m.update({p: "val" for p in va["patient_id"]})
    m.update({p: "test" for p in te["patient_id"]})
    return df["patient_id"].map(m)

if FULL and SPLIT_FILE.exists() and not FORCE_RESPLIT:
    frozen = pd.read_csv(SPLIT_FILE, dtype={"image_id": str})
    all_df = all_df.merge(frozen[["dataset", "image_id", "split"]], on=["dataset", "image_id"], how="left")
    assert all_df["split"].notna().all(), (
        "The frozen split file does not cover all images. It is probably from the OLD 16,051-image subset. "
        "Rename it (for example to split_v1_old_subset.csv) and run again to create a new one.")
    print("Loaded the FROZEN split:", SPLIT_FILE)
else:
    parts = []
    for name, g in all_df.groupby("dataset"):
        g = g.copy()
        g["split"] = patient_level_split(g, SEED)
        parts.append(g)
    all_df = pd.concat(parts, ignore_index=True).sort_values(["dataset", "image_id"]).reset_index(drop=True)
    if FULL:
        SPLIT_DIR.mkdir(parents=True, exist_ok=True)
        all_df[["dataset", "image_id", "patient_id", "label", "split"]].to_csv(SPLIT_FILE, index=False)
        print("Saved NEW frozen split:", SPLIT_FILE)
    else:
        print("Smoke test: split not saved.")

# no patient may appear in two splits
for name, g in all_df.groupby("dataset"):
    n_split = g.groupby("patient_id")["split"].nunique()
    assert (n_split == 1).all(), f"{name}: a patient is in more than one split"
print("Leakage check passed: no patient is in two splits.\n")
for name, g in all_df.groupby("dataset"):
    print(name)
    tbl = pd.crosstab(g["split"], g["label"], margins=True).reindex(["train", "val", "test", "All"])
    display(tbl)
    print((g["split"].value_counts(normalize=True).reindex(["train", "val", "test"]) * 100).round(1).to_dict(), "(% of images)\n")

## 5. Process in shards, then manifest
- Images are cut into **shards** of 2,000 (100 in smoke mode), in a fixed order: APTOS first, then EyePACS.
- A shard is finished when its zip **and** its small `meta` CSV are on Drive. The CSV is written last, so a shard that was cut in half by a disconnect is never counted as finished.
- **Run the notebook again after a disconnect:** finished shards are skipped, only the missing ones are done.
- After every shard the notebook prints the real speed and the time left.
- Output files: 512 x 512 JPEG (quality 95) inside `xretina_<dataset>_512_partNNN.zip`.

In [ ]:
def process_one(row):
    image_id, dataset, src, rel, work = row
    try:
        out, meta = preprocess(src)
        ok = cv2.imwrite(str(work / rel), out, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
        if not ok:
            raise IOError("imwrite failed")
        return dict(image_id=image_id, dataset=dataset, rel_path=rel, ok=True, error="", **meta)
    except Exception as e:
        return dict(image_id=image_id, dataset=dataset, rel_path=rel, ok=False, error=str(e)[:200])

def shard_paths(name, k):
    return (SHARD_DIR / f"xretina_{name}_{OUT_SIZE}_part{k:03d}.zip",
            SHARD_DIR / f"meta_{name}_{OUT_SIZE}_part{k:03d}.csv")

def shard_is_done(name, k, ids):
    z, m = shard_paths(name, k)
    if not (z.exists() and m.exists()):
        return False
    try:
        meta = pd.read_csv(m, dtype={"image_id": str})
    except Exception:
        return False
    if len(meta) != len(ids) or set(meta["image_id"]) != set(ids):
        return False
    if REPROCESS_FAILED_SHARDS and not (meta["ok"] == True).all():
        return False
    return True

SHARD_DIR.mkdir(parents=True, exist_ok=True)
plan = []                                   # (dataset, shard number, rows)
for name in sorted(all_df["dataset"].unique()):       # aptos first, then eyepacs
    sub = all_df[all_df["dataset"] == name].reset_index(drop=True)
    for k, start in enumerate(range(0, len(sub), SHARD_SIZE)):
        plan.append((name, k, sub.iloc[start:start + SHARD_SIZE]))
todo = [p for p in plan if not shard_is_done(p[0], p[1], list(p[2]["image_id"]))]
print(f"{len(plan)} shards in total | {len(plan) - len(todo)} already finished | {len(todo)} to do")
if MAX_SHARDS_THIS_RUN:
    todo = todo[:MAX_SHARDS_THIS_RUN]
    print(f"MAX_SHARDS_THIS_RUN = {MAX_SHARDS_THIS_RUN}: only {len(todo)} shards in this session")

t_start = time.time(); n_img = 0
with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
    for n, (name, k, part) in enumerate(todo, 1):
        t0 = time.time()
        z_final, m_final = shard_paths(name, k)
        work = LOCAL_OUT / "work" / f"{name}_part{k:03d}"
        shutil.rmtree(work, ignore_errors=True)
        (work / name).mkdir(parents=True)
        rows = [(r.image_id, name, r.image_path, f"{name}/{r.image_id}.jpg", work) for r in part.itertuples()]
        res = list(tqdm(ex.map(process_one, rows), total=len(rows), desc=f"{name} part{k:03d}", leave=False))
        meta = pd.DataFrame(res)
        meta["shard"] = z_final.name
        good = meta[meta["ok"] == True]
        local_zip = work / z_final.name
        with zipfile.ZipFile(local_zip, "w", zipfile.ZIP_STORED) as z:
            for rel in good["rel_path"]:
                z.write(work / rel, arcname=rel)
        tmp_zip = z_final.with_name(z_final.name + ".tmp")
        shutil.copy(local_zip, tmp_zip)
        assert tmp_zip.stat().st_size == local_zip.stat().st_size, "zip copy to Drive is incomplete"
        os.replace(tmp_zip, z_final)
        meta.to_csv(m_final, index=False)         # written LAST: this marks the shard as finished
        shutil.rmtree(work, ignore_errors=True)
        n_img += len(part)
        per_shard = (time.time() - t_start) / n
        print(f"[{n}/{len(todo)}] {name} part{k:03d}: {len(part)} images, {len(part) - len(good)} failed, "
              f"{time.time() - t0:.0f} s ({(time.time() - t0) / len(part):.2f} s/img) | "
              f"time left in this run: {per_shard * (len(todo) - n) / 60:.0f} min")

left = [p for p in plan if not shard_is_done(p[0], p[1], list(p[2]["image_id"]))]
ALL_DONE = len(left) == 0
print(f"\nShards finished: {len(plan) - len(left)} of {len(plan)}")
assert ALL_DONE, (f"{len(left)} shards are still missing. This is normal if you stopped early or set "
                  "MAX_SHARDS_THIS_RUN. Run the notebook again: finished shards are skipped.")

In [ ]:
# ---- manifest (only when ALL shards are finished) ----
meta_all = pd.concat([pd.read_csv(shard_paths(n, k)[1], dtype={"image_id": str}) for n, k, _ in plan],
                     ignore_index=True)
manifest = all_df.drop(columns=["image_path"]).merge(meta_all, on=["dataset", "image_id"], how="left")
n_failed = int((manifest["ok"] != True).sum())
manifest_ok = manifest[manifest["ok"] == True].copy()
cols = ["image_id", "dataset", "patient_id", "eye", "label", "split", "rel_path", "shard",
        "orig_w", "orig_h", "crop_x0", "crop_y0", "crop_x1", "crop_y1", "pad_top", "pad_left", "square_side"]
manifest_ok = manifest_ok[cols]
META_DIR.mkdir(parents=True, exist_ok=True)
manifest_ok.to_csv(META_DIR / "manifest.csv", index=False)
(META_DIR / "preprocessing_params.json").write_text(json.dumps(PARAMS, indent=2))
if n_failed:
    print("FAILED images:", n_failed, "-> to retry them set REPROCESS_FAILED_SHARDS = True in the settings cell and run again.")
    display(manifest[manifest["ok"] != True][["dataset", "image_id", "error"]].head(10))
print(f"manifest.csv: {len(manifest_ok)} rows | failed: {n_failed} | saved in {META_DIR}")

## 6. Checks
All checks must pass before the data is handed to Maheen and Amna.

In [ ]:
checks = {}
checks["no failed images"] = (n_failed == 0)
checks["manifest rows == images in this run"] = (len(manifest_ok) == len(all_df))

zips = {}
def _zip(shard):
    if shard not in zips:
        zips[shard] = zipfile.ZipFile(SHARD_DIR / shard)
    return zips[shard]

def read_processed(shard, rel):
    data = np.frombuffer(_zip(shard).read(rel), np.uint8)
    return cv2.imdecode(data, cv2.IMREAD_COLOR)

# every manifest file is inside its shard zip, and the zips hold nothing else
ok_files = True
for shard, g in manifest_ok.groupby("shard"):
    names = _zip(shard).namelist()
    ok_files &= (set(g["rel_path"]) == set(names)) and (len(names) == len(g))
checks["every manifest file is in its shard zip (and nothing extra)"] = bool(ok_files)

leak = manifest_ok.groupby(["dataset", "patient_id"])["split"].nunique()
checks["no patient in two splits (manifest)"] = bool((leak == 1).all())

manifest_ok["kept_frac"] = ((manifest_ok["crop_x1"] - manifest_ok["crop_x0"]) *
                            (manifest_ok["crop_y1"] - manifest_ok["crop_y0"])) / (manifest_ok["orig_w"] * manifest_ok["orig_h"])
odd = manifest_ok[manifest_ok["kept_frac"] < 0.25]
checks["very few odd crops (< 2% of images keep < 25% of the picture)"] = bool(len(odd) <= 0.02 * len(manifest_ok))

smp = manifest_ok.sample(min(200, len(manifest_ok)), random_state=1)
shapes_ok, means = True, []
for r in smp.itertuples():
    im = read_processed(r.shard, r.rel_path)
    shapes_ok &= (im is not None and im.shape == (OUT_SIZE, OUT_SIZE, 3))
    if im is not None:
        means.append(float(im.mean()))
checks[f"sample images are {OUT_SIZE}x{OUT_SIZE}x3"] = bool(shapes_ok)
checks["mean brightness is sensible (20-200)"] = bool(means) and 20 < np.mean(means) < 200

for k, v in checks.items():
    print("PASS" if v else "FAIL", "-", k)
CHECKS_PASSED = all(checks.values())
print("\nALL CHECKS PASSED" if CHECKS_PASSED else "\nSOME CHECKS FAILED - do not hand over the data yet")
print(f"Images with kept area < 25%: {len(odd)}")
if len(odd):
    display(odd[["dataset", "image_id", "label", "kept_frac"]].sort_values("kept_frac").head(8))

show = manifest_ok.sample(min(10, len(manifest_ok)), random_state=3)
fig, ax = plt.subplots(2, 5, figsize=(20, 8))
for a, r in zip(ax.ravel(), show.itertuples()):
    a.imshow(cv2.cvtColor(read_processed(r.shard, r.rel_path), cv2.COLOR_BGR2RGB))
    a.set_title(f"{r.dataset} | grade {r.label} | {r.split}"); a.axis("off")
plt.tight_layout(); plt.show()

final_tbl = pd.crosstab([manifest_ok["dataset"], manifest_ok["split"]], manifest_ok["label"], margins=True)
display(final_tbl)
summary = dict(mode=MODE, params=PARAMS, n_images=int(len(manifest_ok)), n_failed=n_failed,
               n_shards=len(plan), checks={k: bool(v) for k, v in checks.items()},
               counts={f"{d}|{s}": int(n) for (d, s), n in manifest_ok.groupby(["dataset", "split"]).size().items()})
if FULL:
    (OUT_DIR / "preprocessing_summary.json").write_text(json.dumps(summary, indent=2))
    print("Saved:", OUT_DIR / "preprocessing_summary.json")

## 7. Nothing is deleted
The raw datasets stay on Drive (`Datasets/APTOS`, `Datasets/EyePACS`) as the main copy. This notebook writes only to `XRetina_Shared/processed`, `splits` and `outputs`. The shard zips take about 3 GB in total.

## For Maheen and Amna: load the processed data
Paste this in your notebook after the bootstrap cell. It copies nothing to Drive.

```python
import zipfile, glob, pandas as pd
P = "/content/drive/MyDrive/FYP(Gr2-Fall 2026)/XRetina_Shared/processed"
for z in sorted(glob.glob(f"{P}/xretina_*_512_part*.zip")):
    zipfile.ZipFile(z).extractall("/content/data/processed")
man = pd.read_csv(f"{P}/manifest.csv", dtype={"image_id": str})
man["path"] = "/content/data/processed/" + man["rel_path"]
train, val, test = (man[man.split == s] for s in ("train", "val", "test"))
```

Rules for everyone:
- Use the `split` column as it is. Never re-split.
- Do not oversample or augment inside these files. Augmentation, Focal Loss and SMOTE happen in the training code, on the **train** rows only.
- Images are 512 x 512 JPEG. Read with OpenCV (BGR) or PIL (RGB), then resize to 380 (or 224 for the like-for-like run with [11]) in the dataloader.
- `dataset` is `aptos` or `eyepacs`: report results per dataset.